# Detección de objetos con Torchvision
### Faster R-CNN, RetinaNet y SSD: dos filosofías de detección

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Autor: Diego Fischer

En este cuaderno exploraremos la **detección de objetos** usando tres arquitecturas clásicas disponibles en `torchvision.models.detection`, todas con pesos preentrenados sobre el dataset **COCO** (Common Objects in Context, 80 clases de objetos cotidianos):

| Modelo | Familia | Año | Idea central |
|---|---|---|---|
| **Faster R-CNN** | *Two-stage* | 2015 | Primero propone regiones, luego clasifica |
| **SSD** | *One-stage* | 2016 | Detecta en una sola pasada, a múltiples escalas |
| **RetinaNet** | *One-stage* | 2017 | Una sola pasada + *Focal Loss* para el desbalance |

> **¿Por qué estos tres y no YOLO?** YOLO es probablemente el detector más famoso, pero justamente por eso conviene conocer a sus "parientes": Faster R-CNN representa una filosofía completamente distinta (*two-stage*), y RetinaNet muestra cómo la comunidad resolvió el problema que hacía que los detectores *one-stage* fueran menos precisos.

El cuaderno tiene **dos partes**:

- **Parte A — Usar (inferencia):** aplicamos los tres modelos ya entrenados y los comparamos (Fases 1 a 8).
- **Parte B — Entrenar (fine-tuning):** cargamos **tus propias imágenes con labels** y entrenamos cada uno de los tres para detectar **tus** objetos (Fases 9 a 13).

---

## ¿Qué es detectar un objeto?

Antes de escribir código, aclaremos la tarea. En visión por computador hay una escalera de problemas de complejidad creciente:

1. **Clasificación:** ¿qué hay en la imagen? → *"un perro"*
2. **Detección:** ¿qué hay y **dónde** está? → *"un perro en este rectángulo, una persona en este otro"*
3. **Segmentación:** ¿qué píxeles exactos pertenecen a cada objeto?

La detección entrega, para cada objeto encontrado, tres cosas:

- Un **bounding box**: rectángulo `(x1, y1, x2, y2)` que encierra al objeto.
- Una **etiqueta**: la clase del objeto (perro, auto, semáforo...).
- Un **puntaje de confianza** entre 0 y 1: qué tan seguro está el modelo.

### Two-stage vs one-stage: la gran división

Los detectores modernos se agrupan en dos familias según **cuántas pasadas** hacen sobre la imagen para decidir:

**Detectores *two-stage* (dos etapas)** — como Faster R-CNN:
1. Una primera red (la *Region Proposal Network*, RPN) mira la imagen y propone regiones que *podrían* contener algo interesante: "revisa aquí, aquí y aquí".
2. Una segunda etapa toma cada región propuesta y la clasifica con cuidado: "esto es un perro con 0.97 de confianza".

La analogía: es como un control de seguridad en el aeropuerto con **pre-filtro**. Primero un escáner rápido marca las maletas sospechosas, y luego un inspector humano revisa en detalle solo esas maletas. Es más lento, pero muy preciso.

**Detectores *one-stage* (una etapa)** — como SSD, RetinaNet y también YOLO:
- Una sola red mira la imagen completa y, en **una sola pasada**, predice simultáneamente dónde hay objetos y qué son.

La analogía: el inspector revisa **todas** las maletas de una vez, sin pre-filtro. Es mucho más rápido, pero históricamente era menos preciso... hasta que RetinaNet descubrió por qué.

### El problema del desbalance y la Focal Loss

¿Por qué los one-stage eran menos precisos? Porque al mirar toda la imagen de una vez, el modelo evalúa **decenas de miles de ubicaciones posibles**, y la inmensa mayoría son fondo (cielo, pared, pasto). El entrenamiento se ahogaba en ejemplos fáciles de "aquí no hay nada", y los pocos objetos reales quedaban diluidos.

RetinaNet (2017) introdujo la **Focal Loss**: una función de pérdida que le baja el volumen a los ejemplos fáciles ya bien clasificados y concentra el aprendizaje en los casos difíciles. Con ese único cambio, un detector one-stage alcanzó (y superó) la precisión de los two-stage de su época. Es uno de los mejores ejemplos de cómo **diagnosticar bien un problema vale más que agrandar la red**.

---

# PARTE A — Usar los modelos (inferencia)

## Fase 1 — Preparación del entorno

Todo lo que necesitamos viene preinstalado en Colab: PyTorch, Torchvision, Matplotlib y PIL. Solo verificamos versiones y si tenemos GPU disponible.

> **Tip:** aunque este cuaderno corre en CPU, la inferencia (y sobre todo el entrenamiento de la Parte B) es mucho más fluida con GPU. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.

In [ ]:
import torch
import torchvision
from torchvision.models import detection
from torchvision.transforms.functional import to_tensor
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests, time, io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'Dispositivo : {device}')

## Fase 2 — Imágenes de prueba

Usaremos imágenes de dominio público con escenas variadas. Escenas "pobladas" son ideales para comparar detectores, porque exigen encontrar objetos de distintos tamaños y parcialmente ocluidos.

In [ ]:
# Imágenes de prueba (dominio público / licencias abiertas)
URLS = {
    'calle':    'https://images.pexels.com/photos/1004409/pexels-photo-1004409.jpeg?w=1200',
    'perros':   'https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg?w=1200',
    'ciclista': 'https://images.pexels.com/photos/2158963/pexels-photo-2158963.jpeg?w=1200',
}

def descargar_imagen(url):
    """Descarga una imagen desde una URL y la retorna como PIL.Image en RGB."""
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert('RGB')

imagenes = {nombre: descargar_imagen(url) for nombre, url in URLS.items()}

fig, axes = plt.subplots(1, len(imagenes), figsize=(18, 6))
for ax, (nombre, img) in zip(axes, imagenes.items()):
    ax.imshow(img)
    ax.set_title(f'{nombre}  ({img.size[0]}x{img.size[1]} px)')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Fase 3 — Cargar los tres modelos

Aquí está la gracia de `torchvision.models.detection`: cada detector se carga en **una línea**, con pesos ya entrenados sobre COCO. Los tres comparten la misma interfaz de entrada y salida, lo que hace trivial compararlos.

- `weights='DEFAULT'` descarga automáticamente los mejores pesos disponibles.
- `.eval()` pone el modelo en **modo inferencia** (en modo entrenamiento estos modelos esperan también las anotaciones y retornan pérdidas, no predicciones).
- Las clases de COCO vienen en `weights.meta['categories']`.

In [ ]:
# --- Faster R-CNN (two-stage): ResNet-50 + FPN como backbone ---
pesos_frcnn = detection.FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
modelo_frcnn = detection.fasterrcnn_resnet50_fpn_v2(weights=pesos_frcnn)
modelo_frcnn.eval().to(device)

# --- RetinaNet (one-stage con Focal Loss): mismo backbone, comparación justa ---
pesos_retina = detection.RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
modelo_retina = detection.retinanet_resnet50_fpn_v2(weights=pesos_retina)
modelo_retina.eval().to(device)

# --- SSD (one-stage clásico): backbone VGG-16, entrada fija de 300x300 ---
pesos_ssd = detection.SSD300_VGG16_Weights.DEFAULT
modelo_ssd = detection.ssd300_vgg16(weights=pesos_ssd)
modelo_ssd.eval().to(device)

# Las categorías de COCO
CLASES_COCO = pesos_frcnn.meta['categories']

print(f'Modelos cargados en {device}')
print(f'Número de categorías COCO: {len(CLASES_COCO)}')
print(f'Algunas clases: {CLASES_COCO[1:11]}')

> **Observación pedagógica:** elegimos las versiones con backbone **ResNet-50 + FPN** tanto para Faster R-CNN como para RetinaNet. Al usar el mismo "esqueleto" extractor de características, cualquier diferencia de resultados se debe a la **arquitectura de detección** (two-stage vs one-stage), no al backbone. SSD usa VGG-16 y entrada fija de 300×300 px porque así fue concebido: es el representante "histórico" de la familia one-stage.

## Fase 4 — Funciones de inferencia y visualización

Los tres modelos comparten el mismo contrato de Torchvision:

- **Entrada:** una lista de tensores `[C, H, W]` con valores en `[0, 1]`.
- **Salida:** una lista de diccionarios, uno por imagen, con `boxes` (cajas `(x1,y1,x2,y2)`), `labels` (índice de clase) y `scores` (confianza, ordenada de mayor a menor).

In [ ]:
def detectar(modelo, imagen_pil):
    """Ejecuta un detector sobre una imagen PIL.

    Retorna el diccionario de predicciones (boxes, labels, scores)
    y el tiempo de inferencia en segundos.
    """
    tensor = to_tensor(imagen_pil).to(device)  # [C, H, W] en [0, 1]

    with torch.no_grad():  # sin gradientes: solo inferencia
        inicio = time.perf_counter()
        predicciones = modelo([tensor])[0]  # lista de 1 imagen -> primer dict
        if device.type == 'cuda':
            torch.cuda.synchronize()  # esperar a que la GPU termine de verdad
        duracion = time.perf_counter() - inicio

    predicciones = {k: v.cpu() for k, v in predicciones.items()}
    return predicciones, duracion

In [ ]:
def dibujar_detecciones(imagen_pil, predicciones, clases=CLASES_COCO,
                        umbral=0.5, titulo='', ax=None, color='lime'):
    """Dibuja las cajas con score >= umbral sobre la imagen.

    'clases' es la lista/dict que traduce el índice de etiqueta a nombre;
    por defecto usa COCO, pero en la Parte B le pasaremos nuestras clases.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    ax.imshow(imagen_pil)
    n_detecciones = 0

    for caja, etiqueta, score in zip(predicciones['boxes'],
                                     predicciones['labels'],
                                     predicciones['scores']):
        if score < umbral:
            continue
        n_detecciones += 1
        x1, y1, x2, y2 = caja.tolist()
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, y1 - 5,
                f'{clases[etiqueta]} {score:.2f}',
                fontsize=9, color='black',
                bbox=dict(facecolor=color, alpha=0.8, pad=1))

    ax.set_title(f'{titulo}  |  {n_detecciones} detecciones (umbral {umbral})')
    ax.axis('off')
    return n_detecciones

### El umbral de confianza: la perilla que tú controlas

El modelo entrega **todas** sus detecciones, incluso las dudosas. El parámetro `umbral` decide cuáles mostramos:

- **Umbral alto (ej. 0.8):** solo detecciones muy seguras → menos falsos positivos, pero podemos perder objetos difíciles.
- **Umbral bajo (ej. 0.3):** aparece casi todo → encontramos más objetos reales, pero también más "fantasmas".

No existe el umbral "correcto": depende de la aplicación. Un sistema de frenado de emergencia prefiere umbral bajo; un contador automático de inventario prefiere umbral alto.

## Fase 5 — Primer detector en acción: Faster R-CNN

Partimos con el representante *two-stage*. Internamente, al ejecutar el modelo ocurren cuatro pasos: el **backbone** extrae características, la **RPN** propone ~1000 regiones, la **segunda etapa** clasifica y afina cada una, y **NMS** elimina cajas duplicadas.

In [ ]:
predicciones_frcnn, t_frcnn = detectar(modelo_frcnn, imagenes['calle'])

fig, ax = plt.subplots(figsize=(12, 8))
dibujar_detecciones(imagenes['calle'], predicciones_frcnn, umbral=0.6,
                    titulo=f'Faster R-CNN ({t_frcnn:.2f} s)', ax=ax)
plt.show()

print('Top 5 detecciones:')
for i in range(5):
    clase = CLASES_COCO[predicciones_frcnn['labels'][i]]
    score = predicciones_frcnn['scores'][i]
    print(f'  {i+1}. {clase:<15} confianza = {score:.3f}')

## Fase 6 — Los one-stage: RetinaNet y SSD

Ahora las mismas imágenes con los detectores de una sola pasada. **RetinaNet** debería rendir parecido a Faster R-CNN (comparte backbone + Focal Loss); **SSD**, al trabajar con la imagen reducida a 300×300, será el más veloz pero le costarán los objetos pequeños.

In [ ]:
predicciones_retina, t_retina = detectar(modelo_retina, imagenes['calle'])
predicciones_ssd, t_ssd = detectar(modelo_ssd, imagenes['calle'])

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
dibujar_detecciones(imagenes['calle'], predicciones_retina, umbral=0.5,
                    titulo=f'RetinaNet ({t_retina:.2f} s)', ax=axes[0],
                    color='orange')
dibujar_detecciones(imagenes['calle'], predicciones_ssd, umbral=0.5,
                    titulo=f'SSD300 ({t_ssd:.2f} s)', ax=axes[1],
                    color='cyan')
plt.tight_layout()
plt.show()

> **Pregunta guía:** ¿cuál encontró más objetos pequeños o lejanos? Prueba bajar el umbral de SSD a 0.3 y observa si los objetos aparecen (con baja confianza) o simplemente no están.

## Fase 7 — Comparación sistemática

Una imagen no es evidencia. Corramos los tres modelos sobre **todas** las imágenes y tabulemos tiempo y cantidad de detecciones sobre el mismo umbral.

In [ ]:
UMBRAL = 0.5
modelos = {
    'Faster R-CNN': (modelo_frcnn, 'lime'),
    'RetinaNet':    (modelo_retina, 'orange'),
    'SSD300':       (modelo_ssd, 'cyan'),
}

resultados = []  # (imagen, modelo, tiempo, n_detecciones)

for nombre_img, img in imagenes.items():
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for ax, (nombre_mod, (modelo, color)) in zip(axes, modelos.items()):
        pred, t = detectar(modelo, img)
        n = dibujar_detecciones(img, pred, umbral=UMBRAL,
                                titulo=f'{nombre_mod} ({t:.2f} s)',
                                ax=ax, color=color)
        resultados.append((nombre_img, nombre_mod, t, n))
    plt.suptitle(f'Imagen: {nombre_img}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tabla resumen
print(f"{'Imagen':<12}{'Modelo':<15}{'Tiempo (s)':<12}{'Detecciones':<12}")
print('-' * 51)
for nombre_img, nombre_mod, t, n in resultados:
    print(f'{nombre_img:<12}{nombre_mod:<15}{t:<12.3f}{n:<12}')

print()
print('Tiempo promedio por modelo:')
for nombre_mod in modelos:
    tiempos = [t for _, m, t, _ in resultados if m == nombre_mod]
    print(f'  {nombre_mod:<15}{sum(tiempos)/len(tiempos):.3f} s')

> **Advertencia metodológica:** contar detecciones **no** mide calidad — un modelo podría "detectar más" inventando objetos. La métrica estándar del área es el **mAP** (*mean Average Precision*), que compara contra anotaciones reales usando el solapamiento **IoU** entre cajas. Como aquí usamos imágenes sin anotar, la comparación es cualitativa. (En la Parte B sí tendremos anotaciones propias.)

---
# PARTE B — Entrena tu propio detector (fine-tuning)

Hasta aquí usamos modelos que ya sabían detectar las 80 clases de COCO. Pero, ¿y si quieres detectar **tus** objetos: una plaga en hojas de vid, defectos en una pieza, tu mascota, señalética chilena? Para eso entrenamos.

No partiremos de cero: haremos ***transfer learning*** (o *fine-tuning*). La idea es aprovechar que el modelo **ya aprendió a ver** (bordes, texturas, formas) sobre millones de imágenes de COCO, y solo re-entrenar la parte que **decide las clases**. Con eso puedes lograr un detector decente con **decenas o cientos** de imágenes, en vez de los cientos de miles que exigiría entrenar desde cero.

El plan de la Parte B:

- **Fase 9:** el formato que deben tener tus *labels* (¡lee esto con calma!).
- **Fase 10:** cargar tus imágenes + labels y **verificarlos** visualmente.
- **Fase 11:** adaptar cada uno de los tres modelos a tu número de clases.
- **Fase 12:** el entrenamiento propiamente tal.
- **Fase 13:** probar y guardar tu detector entrenado.

> Recomendación fuerte: activa la **GPU** antes de esta parte (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*). Entrenar en CPU es viable para el mini-ejemplo, pero lento para datos reales.

## Fase 9 — El formato de los labels

Un *label* (etiqueta) de detección responde, para cada objeto de cada imagen, a dos preguntas: **qué es** (la clase) y **dónde está** (el bounding box). Todo lo demás es cómo empaquetar esa información en archivos.

### El sistema de coordenadas

En imágenes, el origen `(0, 0)` está **arriba a la izquierda**. `x` crece hacia la derecha y `y` crece hacia **abajo** (¡al revés que en matemáticas!). Un bounding box se describe con dos esquinas:

```
    (x1,y1) ┌───────────┐        x1,y1 = esquina superior izquierda
            │           │        x2,y2 = esquina inferior derecha
            │   objeto  │
            │           │        Siempre: x1 < x2  y  y1 < y2
            └───────────┘ (x2,y2)  Coordenadas en PÍXELES (no normalizadas)
```

### El formato que usaremos: un CSV simple

Hay muchos formatos estándar (los vemos abajo), pero para aprender usaremos el más transparente posible: **un solo archivo `labels.csv`** con una fila por objeto. Puedes crearlo incluso en Excel o Google Sheets.

| filename | x1 | y1 | x2 | y2 | class |
|---|---|---|---|---|---|
| img_001.jpg | 34 | 50 | 200 | 320 | gato |
| img_001.jpg | 210 | 80 | 400 | 300 | perro |
| img_002.jpg | 15 | 15 | 150 | 150 | gato |

Reglas del formato:

- **`filename`**: nombre del archivo de imagen (debe existir en la carpeta de imágenes).
- **`x1, y1, x2, y2`**: las dos esquinas del box, en **píxeles enteros**, respecto al tamaño **original** de esa imagen.
- **`class`**: el nombre de la clase, como texto (`gato`, `perro`, `tornillo`...).
- **Una fila por objeto.** Si una imagen tiene 3 objetos, aparece en 3 filas. Si tiene 1, en 1 fila.
- Una imagen **sin objetos** simplemente no aparece en el CSV (o puedes omitirla).

### La convención del "fondo" (importante)

Internamente, torchvision reserva el índice de clase **0 para el fondo** (*background*). Por eso, al preparar los modelos, el número de clases que declararemos es **tus clases + 1**. Nosotros nos encargamos de esa traducción automáticamente: tú solo escribes nombres de clase en el CSV, y el código asigna `gato→1`, `perro→2`, etc., dejando el `0` para el fondo. Tenlo presente porque es una fuente clásica de bugs.

### Otros formatos estándar (para que los reconozcas)

Si usas una herramienta de anotación real, probablemente exporte en uno de estos. Todos guardan la misma información (clase + caja), con distinta envoltura:

- **Pascal VOC:** un archivo `.xml` por imagen, con las cajas como `xmin, ymin, xmax, ymax` en píxeles. Es el que exporta *LabelImg*.
- **COCO:** un único `.json` para todo el dataset; las cajas van como `[x, y, ancho, alto]` (¡ojo, esquina + tamaño, no dos esquinas!). Es el estándar de la industria y lo exportan *Roboflow* y *CVAT*.
- **YOLO:** un `.txt` por imagen; cada línea es `clase_id x_centro y_centro ancho alto`, todo **normalizado** entre 0 y 1 (dividido por el tamaño de la imagen).

> **Herramientas para anotar tus imágenes gratis:** *LabelImg*, *makesense.ai* (en el navegador, sin instalar), *CVAT* o *Roboflow*. Anota en el formato que prefieras y luego conviértelo al CSV de arriba — la conversión es aritmética simple (por ejemplo, de COCO: `x2 = x + ancho`, `y2 = y + alto`).

## Fase 10 — Cargar tus datos (y verificarlos)

Tienes dos caminos. Elige uno con la variable `USAR_DATOS_DEMO`:

- **`True` (demo):** generamos un mini-dataset sintético de figuras (círculos y cuadrados) para que **todo el pipeline corra de inmediato**, sin que tengas que subir nada. Ideal para ver el flujo completo en clase.
- **`False` (tus datos):** sube un archivo **ZIP** que contenga tus imágenes y un `labels.csv` con el formato de la Fase 9.

Empecemos generando (o cargando) los datos.

In [ ]:
import os, random, zipfile, shutil
import pandas as pd
from PIL import ImageDraw

USAR_DATOS_DEMO = True   # <-- cambia a False para usar TUS datos
CARPETA_DATOS = 'mis_datos'


def generar_datos_demo(carpeta, n_imagenes=20, size=256, seed=0):
    """Crea imágenes sintéticas con círculos y cuadrados y su labels.csv.

    Sirve como dataset de juguete para probar el pipeline de punta a punta.
    """
    random.seed(seed)
    if os.path.exists(carpeta):
        shutil.rmtree(carpeta)
    os.makedirs(carpeta)
    filas = []
    for i in range(n_imagenes):
        img = Image.new('RGB', (size, size), (245, 245, 245))
        dib = ImageDraw.Draw(img)
        for _ in range(random.randint(1, 2)):
            lado = random.randint(45, 85)
            x1 = random.randint(5, size - lado - 5)
            y1 = random.randint(5, size - lado - 5)
            x2, y2 = x1 + lado, y1 + lado
            color = tuple(random.randint(0, 160) for _ in range(3))
            forma = random.choice(['circulo', 'cuadrado'])
            if forma == 'circulo':
                dib.ellipse([x1, y1, x2, y2], fill=color)
            else:
                dib.rectangle([x1, y1, x2, y2], fill=color)
            filas.append([f'img_{i:03d}.png', x1, y1, x2, y2, forma])
        img.save(os.path.join(carpeta, f'img_{i:03d}.png'))
    df = pd.DataFrame(filas, columns=['filename', 'x1', 'y1', 'x2', 'y2', 'class'])
    df.to_csv(os.path.join(carpeta, 'labels.csv'), index=False)
    return df


if USAR_DATOS_DEMO:
    df_labels = generar_datos_demo(CARPETA_DATOS)
    print('Datos de demostración generados en la carpeta:', CARPETA_DATOS)
else:
    from google.colab import files
    print('Sube un .zip con tus imágenes y tu labels.csv ...')
    subido = files.upload()
    nombre_zip = list(subido.keys())[0]
    if os.path.exists(CARPETA_DATOS):
        shutil.rmtree(CARPETA_DATOS)
    os.makedirs(CARPETA_DATOS)
    with zipfile.ZipFile(io.BytesIO(subido[nombre_zip])) as z:
        z.extractall(CARPETA_DATOS)
    # Buscar el labels.csv aunque venga dentro de una subcarpeta
    ruta_csv = None
    for raiz, _, archivos in os.walk(CARPETA_DATOS):
        if 'labels.csv' in archivos:
            ruta_csv = os.path.join(raiz, 'labels.csv')
            CARPETA_DATOS = raiz  # las imágenes están junto al CSV
            break
    assert ruta_csv is not None, 'No encontré labels.csv dentro del ZIP'
    df_labels = pd.read_csv(ruta_csv)

print(f'Filas (objetos) : {len(df_labels)}')
print(f'Imágenes        : {df_labels.filename.nunique()}')
print(f'Clases          : {sorted(df_labels["class"].unique())}')
df_labels.head()

### Construir el mapa de clases y el `Dataset`

PyTorch entrena a partir de un objeto `Dataset` que, por cada imagen, entrega la imagen como tensor y un `target` con `boxes` y `labels`. Aquí traducimos los nombres de clase a índices (recordando: **0 = fondo**, tus clases desde 1).

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Mapa nombre -> índice (0 reservado para el fondo)
nombres_clase = sorted(df_labels['class'].unique())
clase_a_idx = {nombre: i + 1 for i, nombre in enumerate(nombres_clase)}
idx_a_clase = {0: '__fondo__', **{i: n for n, i in clase_a_idx.items()}}
NUM_CLASES = len(nombres_clase) + 1  # +1 por el fondo

print('Traducción de clases:', clase_a_idx)
print('NUM_CLASES (incluye fondo):', NUM_CLASES)


class DatasetDeteccion(Dataset):
    def __init__(self, df, carpeta, clase_a_idx):
        self.carpeta = carpeta
        self.clase_a_idx = clase_a_idx
        self.archivos = sorted(df['filename'].unique())
        # Agrupamos las cajas por imagen para acceso rápido
        self.por_imagen = {a: df[df['filename'] == a] for a in self.archivos}

    def __len__(self):
        return len(self.archivos)

    def __getitem__(self, i):
        nombre = self.archivos[i]
        img = Image.open(os.path.join(self.carpeta, nombre)).convert('RGB')
        filas = self.por_imagen[nombre]
        cajas = filas[['x1', 'y1', 'x2', 'y2']].values.tolist()
        etiquetas = [self.clase_a_idx[c] for c in filas['class']]
        target = {
            'boxes': torch.tensor(cajas, dtype=torch.float32),
            'labels': torch.tensor(etiquetas, dtype=torch.int64),
        }
        return to_tensor(img), target  # imagen en [0,1], target dict


# collate_fn: en detección cada imagen puede tener distinto tamaño y nº de cajas,
# así que NO se apilan en un tensor; se agrupan como listas.
def agrupar(batch):
    return tuple(zip(*batch))

dataset = DatasetDeteccion(df_labels, CARPETA_DATOS, clase_a_idx)
loader = DataLoader(dataset, batch_size=4, shuffle=True,
                    collate_fn=agrupar, num_workers=0)

print(f'Dataset listo: {len(dataset)} imágenes.')

### Verifica tus labels ANTES de entrenar

Este paso es el más subestimado y el más importante. **Labels malos = modelo malo**, sin excepción. Dibujemos las cajas del *ground truth* sobre algunas imágenes para confirmar que las coordenadas y las clases están bien. Si aquí las cajas no calzan con los objetos, no sigas: arregla el CSV primero.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, i in zip(axes, range(min(4, len(dataset)))):
    img_tensor, target = dataset[i]
    img = img_tensor.permute(1, 2, 0).numpy()  # [C,H,W] -> [H,W,C]
    ax.imshow(img)
    for caja, etiqueta in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = caja.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                     linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, y1 - 4, idx_a_clase[int(etiqueta)], fontsize=9,
                color='white', bbox=dict(facecolor='red', alpha=0.8, pad=1))
    ax.set_title(f'{dataset.archivos[i]}')
    ax.axis('off')
plt.suptitle('Verificación de labels (ground truth)', fontsize=14)
plt.tight_layout()
plt.show()

## Fase 11 — Adaptar cada modelo a tus clases

Un modelo preentrenado en COCO tiene su **cabeza de clasificación** dimensionada para 91 categorías. Para detectar *tus* clases hay que **reemplazar esa cabeza** por una nueva del tamaño correcto (`NUM_CLASES`), conservando el resto (el backbone que ya sabe ver).

Aquí aparece un detalle bonito: **cada arquitectura reemplaza su cabeza de forma distinta**, y eso refleja su diseño:

- **Faster R-CNN** tiene un predictor sobre las regiones (`FastRCNNPredictor`): se cambia esa capa final.
- **RetinaNet** tiene una sub-red de clasificación densa (`RetinaNetClassificationHead`) que aplica Focal Loss; se reemplaza cuidando el `norm_layer` (usa GroupNorm).
- **SSD** predice a **varias escalas** a la vez, así que su cabeza (`SSDClassificationHead`) necesita saber los canales y anclas de *cada* nivel.

La función siguiente encapsula las tres cirugías detrás de una interfaz única.

In [ ]:
from functools import partial
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection import _utils as det_utils


def preparar_modelo(nombre, num_clases):
    """Carga un detector preentrenado en COCO y reemplaza su cabeza
    de clasificación para 'num_clases' (incluyendo el fondo).

    nombre: 'fasterrcnn' | 'retinanet' | 'ssd'
    """
    if nombre == 'fasterrcnn':
        modelo = detection.fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
        in_feat = modelo.roi_heads.box_predictor.cls_score.in_features
        modelo.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_clases)

    elif nombre == 'retinanet':
        modelo = detection.retinanet_resnet50_fpn_v2(weights='DEFAULT')
        n_anclas = modelo.head.classification_head.num_anchors
        modelo.head.classification_head = RetinaNetClassificationHead(
            in_channels=256, num_anchors=n_anclas, num_classes=num_clases,
            norm_layer=partial(torch.nn.GroupNorm, 32))

    elif nombre == 'ssd':
        modelo = detection.ssd300_vgg16(weights='DEFAULT')
        canales = det_utils.retrieve_out_channels(modelo.backbone, (300, 300))
        n_anclas = modelo.anchor_generator.num_anchors_per_location()
        modelo.head.classification_head = SSDClassificationHead(
            canales, n_anclas, num_clases)

    else:
        raise ValueError("nombre debe ser 'fasterrcnn', 'retinanet' o 'ssd'")

    return modelo

print('Función preparar_modelo() lista para los tres detectores.')

## Fase 12 — Entrenar

Ahora el bucle de entrenamiento. La clave conceptual: en **modo `train()`**, estos modelos reciben `(imagenes, targets)` y **retornan un diccionario de pérdidas** (no predicciones). Sumamos esas pérdidas, propagamos el gradiente hacia atrás y actualizamos los pesos. Repetir muchas veces = aprender.

Elige qué arquitectura entrenar cambiando `MODELO_A_ENTRENAR`. Puedes correr esta celda tres veces (una por modelo) para comparar, tal como en la Parte A.

In [ ]:
def entrenar(modelo, loader, epocas=10, lr=0.005):
    """Entrena un detector y devuelve el historial de pérdida por época."""
    modelo.to(device).train()
    params = [p for p in modelo.parameters() if p.requires_grad]
    optimizador = torch.optim.SGD(params, lr=lr, momentum=0.9,
                                  weight_decay=0.0005)
    historial = []
    for epoca in range(epocas):
        suma, n = 0.0, 0
        for imgs, targets in loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            perdidas = modelo(imgs, targets)      # dict de pérdidas
            perdida = sum(perdidas.values())

            optimizador.zero_grad()
            perdida.backward()
            optimizador.step()

            suma += perdida.item()
            n += 1
        promedio = suma / max(n, 1)
        historial.append(promedio)
        print(f'Época {epoca + 1:2d}/{epocas}   pérdida = {promedio:.4f}')
    return historial

In [ ]:
# Elige aquí: 'fasterrcnn' | 'retinanet' | 'ssd'
MODELO_A_ENTRENAR = 'fasterrcnn'
NUM_EPOCAS = 10

mi_modelo = preparar_modelo(MODELO_A_ENTRENAR, NUM_CLASES)
print(f'Entrenando {MODELO_A_ENTRENAR} para {NUM_CLASES} clases '
      f'(incluye fondo) en {device}...\n')

historial = entrenar(mi_modelo, loader, epocas=NUM_EPOCAS, lr=0.005)

# Curva de pérdida
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(historial) + 1), historial, marker='o')
plt.xlabel('Época'); plt.ylabel('Pérdida promedio')
plt.title(f'Curva de entrenamiento — {MODELO_A_ENTRENAR}')
plt.grid(alpha=0.3)
plt.show()

> **Cómo leer la curva:** debería **bajar** y luego aplanarse. Si baja muy lento, sube `NUM_EPOCAS` o el `lr`. Si sube o explota (NaN), baja el `lr`. Con el dataset demo de figuras y 10 épocas verás la pérdida caer, aunque para resultados nítidos conviene más datos y más épocas — es un juguete para validar el flujo, no un modelo de producción.

## Fase 13 — Probar (y guardar) tu detector entrenado

Ponemos el modelo en `eval()` y lo aplicamos sobre una imagen. Reutilizamos las funciones `detectar` y `dibujar_detecciones` de la Parte A, pero ahora pasándole **nuestro** diccionario de clases (`idx_a_clase`) en vez del de COCO.

In [ ]:
mi_modelo.eval()

# Tomamos una imagen del propio dataset para probar
# (con tus datos reales, idealmente usa una imagen que NO estuvo en el entrenamiento)
img_prueba = Image.open(
    os.path.join(CARPETA_DATOS, dataset.archivos[0])).convert('RGB')

pred, t = detectar(mi_modelo, img_prueba)

fig, ax = plt.subplots(figsize=(8, 8))
dibujar_detecciones(img_prueba, pred, clases=idx_a_clase, umbral=0.5,
                    titulo=f'{MODELO_A_ENTRENAR} entrenado', ax=ax, color='deeppink')
plt.show()

### Guardar y volver a cargar los pesos

Cuando tu detector funcione, guarda sus pesos para no re-entrenar cada vez. En Colab puedes descargar el archivo o montar tu Google Drive.

In [ ]:
# Guardar
ruta_pesos = f'detector_{MODELO_A_ENTRENAR}.pth'
torch.save(mi_modelo.state_dict(), ruta_pesos)
print('Pesos guardados en:', ruta_pesos)

# Para volver a cargarlos en otra sesión:
#   modelo = preparar_modelo(MODELO_A_ENTRENAR, NUM_CLASES)
#   modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
#   modelo.eval().to(device)

# Descargar el archivo a tu computador (opcional):
#   from google.colab import files
#   files.download(ruta_pesos)

---
## Reflexión: detección de objetos y responsabilidad

La detección de objetos es probablemente la capacidad de visión por computador con más impacto directo en el mundo físico: vehículos autónomos, videovigilancia, control de calidad industrial, conteo de fauna, diagnóstico por imágenes. Algunas preguntas para discutir:

1. **Sesgos del dataset:** COCO fue construido mayoritariamente con imágenes de contextos occidentales. Y cuando *tú* entrenas con tus propios datos, tus sesgos de muestreo pasan directo al modelo: si solo fotografiaste tus objetos con buena luz y fondo limpio, fallará en condiciones reales. ¿Qué tan representativo es tu dataset?
2. **El costo del error:** un falso negativo en un auto autónomo (no ver a un peatón) y un falso positivo en videovigilancia (marcar a una persona inocente) tienen consecuencias muy distintas. ¿Cómo debería influir eso en el umbral y el modelo?
3. **Vigilancia:** la misma tecnología que cuenta autos en una autopista puede seguir personas en una protesta. Ahora que sabes lo fácil que es entrenar un detector a medida, ¿dónde pones los límites y quién los pone?

---

## Ideas clave (para repasar en clase)

**Sobre usar detectores (Parte A):**

- **Detección = clasificación + localización:** cada predicción trae caja, etiqueta y confianza.
- **Two-stage (Faster R-CNN):** propone regiones y luego clasifica. Preciso, más lento. **One-stage (SSD, RetinaNet, YOLO):** todo en una pasada. Rápido, pero sufre el desbalance fondo/objeto.
- **Focal Loss (RetinaNet):** concentra el aprendizaje en los casos difíciles; cerró la brecha con los two-stage.
- **El umbral de confianza es una decisión de diseño**, no una propiedad del modelo.
- Para comparar en serio se usa **mAP** sobre datos anotados; contar detecciones solo da intuición.

**Sobre entrenar tu propio detector (Parte B):**

- **Transfer learning:** partimos de pesos COCO y solo reemplazamos la **cabeza de clasificación**; así bastan decenas/cientos de imágenes en vez de cientos de miles.
- **Formato de labels:** por objeto, clase + caja `(x1,y1,x2,y2)` en píxeles, origen arriba-izquierda. El índice **0 se reserva para el fondo**, así que `num_classes = tus_clases + 1`.
- **Cada arquitectura reemplaza su cabeza distinto** (FastRCNNPredictor / RetinaNetClassificationHead / SSDClassificationHead), reflejando su diseño.
- **En modo `train()`** los modelos devuelven **pérdidas**, no detecciones; en `eval()`, al revés.
- **Verifica tus labels antes de entrenar:** labels malos = modelo malo, siempre.
- La calidad y representatividad de **tus datos** manda más que la elección de arquitectura.

---
## Licencia

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Esta obra está licenciada bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es). Puede compartirse y adaptarse, incluso con fines comerciales, siempre que se reconozca la autoría de forma adecuada.

**Atribución:** Diego Fischer — Universidad de Santiago de Chile (USACH).